# 3 · Did the run learn—and can it truly continue?

Chapter 6 / Day 8. Integrate the recipe with the Chapter 5 decoder using a bounded 24-update CPU experiment. This is an executable systems audit, not Day 9's larger pretraining campaign.

**Opening question.** If a restarted model has exactly the saved weights, is the next update necessarily the same? State what else could matter.

All saved checkpoints are trusted, locally created temporary files, removed when each temporary-directory context finishes.


In [ ]:
from pathlib import Path
import sys, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import torch
import matplotlib.pyplot as plt
from dongxi_llms import pretraining_lab as lab
from dongxi_llms import pretraining_visuals as viz
from dongxi_llms.decoder_lab import parameter_count
torch.set_num_threads(1)
print("CPU teaching lab", torch.__version__)
# Source notebooks stay unexecuted; all checkpoints below use temporary directories.


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Training system with validation and checkpoint recovery highlighted](../figures/chapter-06/03-system.png)


In [ ]:
fig = viz.process_map([3, 4, 5]); plt.show()


## 1. Validation is a measurement contract

Use the same tokenizer, shift, context policy and loss reduction, but fixed held-out documents and no parameter update. Weight by valid target counts across the entire validation set. `eval()` changes layer behavior; `no_grad()` suppresses autograd recording. They do different jobs.

**Exercise.** Should evaluation with batches of size 1 and 3 give the same answer? Why could averaging their batch means break this?

**Reference solution.** The intended corpus mean is summed NLL / total valid targets. Grouping should not change it, apart from floating-point reduction differences. Unequal tail lengths make an unweighted mean of batch means a different statistic.


In [ ]:
model = lab.make_model(dtype=torch.float64)
_, valid = lab.fixture()
one = lab.evaluate(model, valid, batch=1)
three = lab.evaluate(model, valid, batch=3)
assert abs(one-three) < 1e-12
assert all(p.grad is None for p in model.parameters())
print("Validation NLL with different batch grouping:", one, three)


## 2. A short actual decoder run

**Prediction.** Must validation loss decrease on every update? Is a lower training curve than validation necessarily a bug?

**Reference solution.** Neither monotonicity nor a fixed gap is guaranteed. Training and validation sample different text. Here the training loss is measured on the current batch **before** its update; validation is measured on the full held-out fixture **after** it. The annotated plot keeps that distinction visible.


In [ ]:
session = lab.TrainingSession(total=24)
history = []
for _ in range(24):
    row = session.update()
    row["validation"] = lab.evaluate(session.model, session.valid)
    history.append(row)
assert all(torch.isfinite(torch.tensor([r["loss"], r["validation"], r["grad_norm"]])).all() for r in history)
print("First update:", history[0])
print("Last update:", history[-1])
print("Actual valid targets vs position ceiling:", session.tokens, 24*1*16*2)
assert session.tokens <= 768


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Measured training-batch and held-out validation NLL over 24 updates](../figures/chapter-06/03-validation.png)


In [ ]:
fig = viz.validation_plot(history); plt.show()


The sentences are related, tiny and authored to make the experiment understandable. A finite or falling curve here does not establish linguistic capability, representative generalization, or benchmark quality. We are validating the machinery and the reporting discipline.

## 3. A checkpoint is a state of the process

The helper stores weights, AdamW state, completed-update count, target counter, data order/cursor/epoch, CPU random state, model configuration, data fingerprints, tokenizer identity, and the schedule contract. The learning rate is recomputed from the saved completed-update count, so there is no hidden scheduler object to forget.

**Exercise.** Which omission changes which causal path: optimizer moments, or the data cursor? Predict the two controls separately.

**Reference solution.** Missing moments changes the update even with the same next examples. Missing the cursor changes examples and therefore their gradients. Both can produce finite losses, so “it runs” is a weak recovery test.


In [ ]:
with tempfile.TemporaryDirectory(prefix="day8-checkpoint-") as directory:
    recovery = lab.recovery_audit(Path(directory) / "trusted-checkpoint.pt", total=24)
for name in ("complete", "no_optimizer", "no_cursor"):
    print(name, recovery[name])
assert recovery["complete"]["parameter_error"] == 0.
assert recovery["complete"]["same_history"]
assert recovery["no_optimizer"]["same_batches"]
assert not recovery["no_cursor"]["same_batches"]
assert recovery["no_optimizer"]["parameter_error"] > 1e-5
assert recovery["no_cursor"]["parameter_error"] > 1e-5


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Parameter difference after complete versus incomplete checkpoint recovery](../figures/chapter-06/03-recovery.png)


In [ ]:
fig = viz.recovery_plot(recovery); plt.show()


**Explanation.** The reference runs uninterrupted. Its checkpoint at update 12 is loaded into three fresh sessions, each continuing to update 24. Only complete restoration must match. The equality claim is scoped to this CPU, dropout-free implementation and the tested software environment—not guaranteed across Mac/Spark, devices, PyTorch versions, or distributed loaders.

Saving halfway through gradient accumulation needs additional gradient buffers and microstep state. Our checkpoints are only at completed-update boundaries. A production trainer may also require CUDA/Python/NumPy RNG, rank-specific samplers, worker/prefetch state, scaler state, and atomic checkpoint publication. A checkpoint should be tested by recovery, not merely by checking that a file exists.


## 4. Reject a changed experiment, even if shapes fit

**Exercise.** Why is loading the same-size embedding matrix unsafe if the tokenizer's ID meanings changed?

**Reference solution.** Shapes say how many entries exist, not what each entry means. Resume must preserve the experiment identity. The check below fails before loading weights.


In [ ]:
fresh = lab.TrainingSession()
state = fresh.checkpoint()
state["contract"]["tokenizer"] = "same-size-but-different-id-meaning"
try:
    fresh.restore(state)
except ValueError as error:
    print("Expected contract rejection:", error)
else:
    raise AssertionError("Changed tokenizer was accepted")


## 5. Turn the mechanism into a bounded specification

**Exercise.** Before a larger Spark run, specify the data revision, model configuration, exact token budget, update rule, precision path, validation contract, recovery test, memory reserve, time limit, and failure criteria. Which values have these CPU notebooks actually measured?

**Reference solution.** [The bounded Day 8 recipe](../../experiments/specs/2026-09-09-day8-bounded-pretraining.md) fixes the executable CPU control and names the gates for any later GPU candidate. Notebook measurements establish toy counts, finite updates and recovery behavior; they do not establish GPU throughput, peak unified memory, corpus quality, or an appropriate large-run learning rate.

A useful Day 9 comparison changes one named factor while holding its evaluation contract and declared resource budget fixed. “More steps” changes token exposure; “same steps, larger batch” also changes exposure. State what is equal before declaring a winner.

## Exit discussion

Explain how a run could have decreasing loss, an apparently healthy memory footprint, and still be an invalid experiment. Possible answers: leaked validation, wrong labels, incorrect accumulation denominator, or unrecoverable training state. Build an argument using observations and missing evidence, not one success flag.

Material prepared does not mean the lesson is mastered. Bring your predictions or surprising results back to the conversation; no calculation quiz is required.
